In [6]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

In [8]:
# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv("processed_diabetes.csv")

TARGET = "Diabetes_binary"

X = df.drop(columns=[TARGET])
y = df[TARGET]

In [9]:
# ============================================================
# OPTIONAL HEAVY FEATURE ENGINEERING
# ============================================================
def engineer_features(df):
    df = df.copy()
    df["BMI_SQ"] = df["BMI"] ** 2
    df["AgeRatio"] = df["Age"] / (df["BMI"] + 1)
    df["PhysDietCombo"] = df["PhysActivity"] * df["Fruits"]
    df["BP_Chol"] = df["HighBP"] + df["HighChol"]
    return df

X = engineer_features(X)

In [11]:
# ============================================================
# TRAIN/TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ============================================================
# OPTUNA HYPERPARAMETER SEARCH
# ============================================================
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 2.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10)  # imbalance
    }

    # Stratified 5-fold CV
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for train_idx, valid_idx in kf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=1500)
        model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=False)
    ]
)


        preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, preds)
        aucs.append(auc)

    return np.mean(aucs)


print("\n Running Optuna optimization…")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)  # <-- HEAVY DUTY
best_params = study.best_params

print("\n Best Hyperparameters Found:")
print(best_params)

[I 2025-11-14 20:29:06,805] A new study created in memory with name: no-name-3d5d4509-7ec6-4efe-b26c-17db938c6e42



 Running Optuna optimization…


[I 2025-11-14 20:29:08,859] Trial 0 finished with value: 0.8298403810326846 and parameters: {'num_leaves': 66, 'learning_rate': 0.1595348005451653, 'feature_fraction': 0.9241987020081043, 'bagging_fraction': 0.9983470685639486, 'bagging_freq': 1, 'max_depth': 9, 'min_data_in_leaf': 168, 'lambda_l1': 6.340319558421319, 'lambda_l2': 4.794119615973109, 'min_split_gain': 0.9963374251736212, 'scale_pos_weight': 9.747393868538461}. Best is trial 0 with value: 0.8298403810326846.
[I 2025-11-14 20:29:11,698] Trial 1 finished with value: 0.8290794580676275 and parameters: {'num_leaves': 218, 'learning_rate': 0.19491960420697532, 'feature_fraction': 0.7808966249876639, 'bagging_fraction': 0.6883207702071665, 'bagging_freq': 10, 'max_depth': 16, 'min_data_in_leaf': 140, 'lambda_l1': 4.7293663147463665, 'lambda_l2': 9.913096421172275, 'min_split_gain': 0.5125108546180503, 'scale_pos_weight': 6.869218980915358}. Best is trial 0 with value: 0.8298403810326846.
[I 2025-11-14 20:29:14,583] Trial 2 fin


 Best Hyperparameters Found:
{'num_leaves': 244, 'learning_rate': 0.03663252794474421, 'feature_fraction': 0.8214551309552943, 'bagging_fraction': 0.6111326974055141, 'bagging_freq': 2, 'max_depth': 5, 'min_data_in_leaf': 84, 'lambda_l1': 8.686567522709549, 'lambda_l2': 5.735571175839784, 'min_split_gain': 0.536603807077058, 'scale_pos_weight': 3.0553364164758805}


In [12]:
# ============================================================
# FINAL TRAINING WITH BEST PARAMS + CALIBRATION
# ============================================================
best_model = lgb.LGBMClassifier(
    **best_params,
    objective="binary",
    metric="auc",
    n_estimators=2000
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", CalibratedClassifierCV(best_model, method="sigmoid", cv=5))
])

print("\nTraining final model…")
pipe.fit(X_train, y_train)


Training final model…


,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,estimator,LGBMClassifie...3364164758805)
,method,'sigmoid'
,cv,5
,n_jobs,None


In [14]:

# ============================================================
# EVALUATION
# ============================================================
print("\n Evaluating on test set…")
pred_probs = pipe.predict_proba(X_test)[:, 1]
preds = (pred_probs >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

print("\n ROC AUC:", roc_auc_score(y_test, pred_probs))
print(" F1 Score:", f1_score(y_test, preds))




 Evaluating on test set…

Classification Report:
              precision    recall  f1-score   support

         0.0       0.81      0.66      0.72      7070
         1.0       0.71      0.84      0.77      7069

    accuracy                           0.75     14139
   macro avg       0.76      0.75      0.75     14139
weighted avg       0.76      0.75      0.75     14139


Confusion Matrix:
[[4634 2436]
 [1112 5957]]

 ROC AUC: 0.8274854470472208
 F1 Score: 0.7705342129090674
